# Normalizacao do Relatorio de Receitas do Sistema ATUA

Este notebook le o arquivo `Relatorio_Receitas_Sistema-ATUA_{MM}.xls` (sistema ATUA, GSL Logistica) e converte para o layout do fechamento SAGI (`FECHAMENTO_ODBC_{AAAA}_{MM}.xlsx`).

Ajuste o parametro `MES_REFERENCIA` na primeira celula de codigo (formato `MM/AAAA`).

Cada linha do arquivo e um CTRC (Conhecimento de Transporte Rodoviario de Cargas) — ou seja, uma receita de frete faturada. No SAGI, a GSL esta na divisao **2.4 TRANSMOVE** (lado receita), com tres filiais ativas neste relatorio:

- GSL PRUDENTE -> 2.4.1 PRESIDENTE PRUDENTE -> 2.4.1.1 TRANSPORTE
- GSL DOURADOS -> 2.4.2 DOURADOS -> 2.4.2.1 TRANSPORTE
- GSL MARINGA PR -> 2.4.3 MARINGA -> 2.4.3.1 TRANSPORTE

O Plano de Contas de todas as linhas e **5.7.1 FRETES** (receita de frete proprio).

In [ ]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

# ── Parametros ──────────────────────────────────────────────────────────────
MES_REFERENCIA = "01/2026"   # formato MM/AAAA — ajuste para cada fechamento
ATUA_SUBPASTA = "Jan"
ARQUIVO_ENTRADA_NOME = "Relatorio_Receitas_Sistema-ATUA_01.xls"
MODELO_MES = "03/2026"       # fallback de modelo FECHAMENTO_ODBC
# ────────────────────────────────────────────────────────────────────────────

mes_num, ano = MES_REFERENCIA.split("/")

REFS_DIR = Path("../../02-Referencias")
ATUA_DIR = REFS_DIR / "ATUA"
ATUA_MES_DIR = (ATUA_DIR / ATUA_SUBPASTA) if ATUA_SUBPASTA else ATUA_DIR

if ARQUIVO_ENTRADA_NOME:
    ARQUIVO_ENTRADA = ATUA_MES_DIR / ARQUIVO_ENTRADA_NOME
else:
    ARQUIVO_ENTRADA = ATUA_MES_DIR / f"Relatorio_Receitas_Sistema-ATUA_{mes_num}.xls"

if MODELO_MES:
    _m_mes, _m_ano = MODELO_MES.split("/")
    ARQUIVO_MODELO = REFS_DIR / f"FECHAMENTO_ODBC_{_m_ano}_{_m_mes}.xlsx"
else:
    ARQUIVO_MODELO = REFS_DIR / f"FECHAMENTO_ODBC_{ano}_{mes_num}.xlsx"
if not ARQUIVO_MODELO.exists():
    _mods = sorted(REFS_DIR.glob("FECHAMENTO_ODBC_*.xlsx"))
    if not _mods:
        _mods = sorted((REFS_DIR / "Fechamento").glob("FECHAMENTO_ODBC_*.xlsx"))
    if not _mods:
        _mods = sorted((REFS_DIR / "outros").glob("FECHAMENTO_ODBC_*.xlsx"))
    if not _mods:
        _mods = [
            ATUA_MES_DIR / f"ATUA_despesas_fechamento_{mes_num}-{ano}.xlsx",
            ATUA_MES_DIR / f"ATUA_receitas_fechamento_{mes_num}-{ano}.xlsx",
        ]
        _mods = [p for p in _mods if p.exists()]
    if _mods:
        ARQUIVO_MODELO = _mods[-1]
        print(f"[AVISO] Modelo de {MES_REFERENCIA} nao encontrado; usando {ARQUIVO_MODELO.name}")
    else:
        raise FileNotFoundError(
            f"Nenhum FECHAMENTO_ODBC ou fechamento ATUA de referencia em {REFS_DIR.resolve()}"
        )

ARQUIVO_SAIDA = ATUA_MES_DIR / f"ATUA_receitas_fechamento_{mes_num}-{ano}.xlsx"

if not ARQUIVO_ENTRADA.exists():
    raise FileNotFoundError(f"Arquivo ATUA Receitas nao encontrado: {ARQUIVO_ENTRADA.resolve()}")

ATUA_MES_DIR.mkdir(parents=True, exist_ok=True)

# O arquivo possui pequena corrupcao interna e, em alguns meses, traz uma linha de total
# antes do cabecalho. Detectamos dinamicamente a linha de cabecalho.
raw_receitas = pd.read_excel(
    ARQUIVO_ENTRADA,
    sheet_name=0,
    header=None,
    dtype=object,
    engine="xlrd",
    engine_kwargs={"ignore_workbook_corruption": True},
)

header_idx = None
for i in range(min(20, len(raw_receitas))):
    vals = [str(v).strip().lower() for v in raw_receitas.iloc[i].tolist() if pd.notna(v)]
    if "nm_pessoa_filial" in vals and "vl_frete_empresa" in vals:
        header_idx = i
        break
if header_idx is None:
    header_idx = 0

df_receitas = raw_receitas.iloc[header_idx + 1 :].copy()
df_receitas.columns = [str(c).strip() for c in raw_receitas.iloc[header_idx].tolist()]
df_receitas = df_receitas.reset_index(drop=True)

# Remove linhas completamente vazias
mask_vazia = df_receitas.apply(
    lambda r: all(pd.isna(v) or str(v).strip() == "" for v in r.values), axis=1
)
df_receitas = df_receitas.loc[~mask_vazia].reset_index(drop=True)

import sys
sys.path.insert(0, str((Path.cwd().parent / "Utitlities").resolve()))
from fechamento_excel import normalizar_colunas_data

df_receitas = normalizar_colunas_data(df_receitas, ("dt_emissao",))

print(f"Entrada: {ARQUIVO_ENTRADA.resolve()}")
print(f"Header detectado na linha: {header_idx}")
print(f"Linhas lidas: {len(df_receitas)}")
print(f"Colunas: {df_receitas.columns.tolist()}")
print()
df_receitas.head(5)

## Mapeamento de Centro de Custo (Receita)

O campo `nm_pessoa_filial` no ATUA identifica qual filial GSL emitiu o frete. O mapeamento para a hierarquia SAGI e:

| nm_pessoa_filial | n2     | n3     | n4       | Descricoes                                 |
|------------------|--------|--------|----------|--------------------------------------------|
| GSL PRUDENTE     | 2.4    | 2.4.1  | 2.4.1.1  | TRANSMOVE GSL > PRESIDENTE PRUDENTE > TRANSPORTE |
| GSL DOURADOS     | 2.4    | 2.4.2  | 2.4.2.1  | TRANSMOVE GSL > DOURADOS > TRANSPORTE      |
| GSL MARINGA PR   | 2.4    | 2.4.3  | 2.4.3.1  | TRANSMOVE GSL > MARINGA > TRANSPORTE       |

In [ ]:
def _str(v) -> str:
    if pd.isna(v):
        return ""
    return str(v).strip()

MAPA_FILIAL_RECEITA = {
    "GSL PRUDENTE": {
        "n3_cod":  "2.4.1",
        "n3_desc": "PRESIDENTE PRUDENTE",
        "n4_cod":  "2.4.1.1",
        "n4_desc": "TRANSPORTE",
        "filial_saida": "GSL PRUDENTE",
    },
    "GSL DOURADOS": {
        "n3_cod":  "2.4.2",
        "n3_desc": "DOURADOS",
        "n4_cod":  "2.4.2.1",
        "n4_desc": "TRANSPORTE",
        "filial_saida": "GSL DOURADOS",
    },
    "GSL MARINGA PR": {
        "n3_cod":  "2.4.3",
        "n3_desc": "MARINGA",
        "n4_cod":  "2.4.3.1",
        "n4_desc": "TRANSPORTE",
        "filial_saida": "GSL MARINGA",
    },
}

def mapear_cc_receita(nm_filial) -> dict | None:
    """Retorna hierarquia SAGI completa para a filial, ou None se nao mapeada.

    Hierarquia de 3 niveis uteis (o prefixo numerico nao conta como nivel):
      n1: 2.4       -> TRANSMOVE GSL
      n2: 2.4.X     -> filial (ex: 2.4.1 PRESIDENTE PRUDENTE)
      n3: 2.4.X.Y   -> setor  (ex: 2.4.1.1 TRANSPORTE)
      n4: igual a n3 -> padrao do modelo FECHAMENTO_ODBC
    """
    chave = _str(nm_filial)
    info  = MAPA_FILIAL_RECEITA.get(chave)
    if not info:
        return None
    return {
        "n1_cod":  "2.4",
        "n1_desc": "TRANSMOVE GSL",
        "n2_cod":  info["n3_cod"],
        "n2_desc": info["n3_desc"],
        "n3_cod":  info["n4_cod"],
        "n3_desc": info["n4_desc"],
        "n4_cod":  info["n4_cod"],
        "n4_desc": info["n4_desc"],
        "filial_saida": info["filial_saida"],
        "segmento": "TRANSMOVE GSL",
    }

print("Filiais unicas no arquivo:")
print(df_receitas["nm_pessoa_filial"].value_counts(dropna=False).to_string())
print()

filiais_nao_mapeadas = set()
for v in df_receitas["nm_pessoa_filial"].unique():
    chave = _str(v)
    if chave not in MAPA_FILIAL_RECEITA:
        filiais_nao_mapeadas.add(chave)

if filiais_nao_mapeadas:
    print(f"[AVISO] Filiais NAO mapeadas ({len(filiais_nao_mapeadas)}):")
    for f in sorted(filiais_nao_mapeadas):
        print(f"  '{f}'")
else:
    print("[OK] Todas as filiais estao mapeadas.")

## Plano de Contas — Fixo: 5.7.1 FRETES

Todas as 138 linhas deste relatorio sao CTRCs (fretes proprios da GSL faturados a clientes). No SAGI, a conta de receita correspondente e **5.7.1 FRETES**.

In [ ]:
COD_CONTA_RECEITA  = "5.7.1"
DESC_CONTA_RECEITA = "FRETES"

print(f"Plano de Contas fixo para todas as linhas: {COD_CONTA_RECEITA} {DESC_CONTA_RECEITA}")

## Conversao para o layout FECHAMENTO_ODBC

Regras de mapeamento de colunas:

| Coluna FECHAMENTO_ODBC | Origem ATUA                          | Observacao                                    |
|------------------------|--------------------------------------|-----------------------------------------------|
| filial                 | filial_saida (do mapa CC)            |                                               |
| titulo                 | `CTRC-{nr_ctrc}`                     |                                               |
| credor_forn_cli_func   | nm_pessoa_destinatario               | cliente / tomador do frete                    |
| data_nf                | dt_emissao                           | data de emissao do CTRC                       |
| data_pagamento         | dt_emissao                           | mesma data (nao ha data de pagamento separada) |
| valor_nf               | vl_frete_empresa                     | positivo                                      |
| valor_pago             | vl_frete_empresa                     | negativo                                      |
| valor_conta            | vl_frete_empresa                     | negativo                                      |
| Valor Oficial          | vl_frete_empresa                     | igual a valor_conta                           |
| cod_conta              | `5.7.1`                              |                                               |
| conta                  | `FRETES`                             |                                               |
| observacao             | CTRC {nr_ctrc} - Motorista: ... - Destino: ... |                                      |
| Origem                 | `Saida (Aplicacoes)`                 |                                               |
| Sistema                | `ATUA`                               |                                               |
| n1_cod … n4_desc       | MAPA_FILIAL_RECEITA                  |                                               |

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str((Path.cwd().parent / "Utitlities").resolve()))
from fechamento_excel import parse_valor_fechamento, parse_data_fechamento, gravar_fechamento_excel

def _to_float(v):
    if pd.isna(v):
        return 0.0
    try:
        return float(str(v).strip().replace(",", "."))
    except (ValueError, TypeError):
        return 0.0


# Carrega colunas do modelo para garantir ordem e completude
modelo_cols = pd.read_excel(ARQUIVO_MODELO, nrows=0).columns.tolist()
COL_VO = next((c for c in modelo_cols if str(c).strip() == "Valor Oficial"), "Valor Oficial")

filiais_sem_mapa = []
linhas_saida = []

for _, row in df_receitas.iterrows():
    filial_raw = _str(row.get("nm_pessoa_filial", ""))
    cc = mapear_cc_receita(filial_raw)

    if cc is None:
        filiais_sem_mapa.append(filial_raw)

    ctrc     = _str(row.get("nr_ctrc", ""))
    valor    = _to_float(row.get("vl_frete_empresa", 0))
    destino  = _str(row.get("nm_cidade_destinatario", ""))
    motorist = _str(row.get("nm_pessoa_motorista", ""))
    cliente  = _str(row.get("nm_pessoa_destinatario", ""))

    nova = {col: "" for col in modelo_cols}

    nova["filial"]               = cc["filial_saida"] if cc else filial_raw
    nova["titulo"]               = str(ctrc)
    nova["credor_forn_cli_func"] = cliente
    nova["data_nf"]              = parse_data_fechamento(row.get("dt_emissao", ""))
    nova["data_pagamento"]       = parse_data_fechamento(row.get("dt_emissao", ""))
    nova["valor_nf"]             = valor
    nova["valor_pago"]           = -abs(valor) if valor is not None else pd.NA
    nova["valor_conta"]          = -abs(valor) if valor is not None else pd.NA
    nova[COL_VO]                 = nova["valor_conta"]
    nova["cod_conta"]            = COD_CONTA_RECEITA
    nova["conta"]                = DESC_CONTA_RECEITA
    nova["observacao"]           = destino
    nova["Origem"]               = "Entradas (Origem)"
    nova["Sistema"]              = "ATUA"
    nova["Segmento"]             = cc["segmento"]     if cc else ""
    nova["cod_conta-descr"]      = f"{COD_CONTA_RECEITA} {DESC_CONTA_RECEITA}"
    nova["n1_cod_centro_custo"]  = cc["n1_cod"]       if cc else ""
    nova["n1_centro_custo"]      = cc["n1_desc"]      if cc else ""
    nova["n1_CC"]                = f"{cc['n1_cod']} {cc['n1_desc']}" if cc else ""
    nova["n2_cod_centro_custo"]  = cc["n2_cod"]       if cc else ""
    nova["n2_centro_custo"]      = cc["n2_desc"]      if cc else ""
    nova["n2_CC"]                = f"{cc['n2_cod']} {cc['n2_desc']}" if cc else ""
    nova["n3_cod_centro_custo"]  = cc["n3_cod"]       if cc else ""
    nova["n3_centro_custo"]      = cc["n3_desc"]      if cc else ""
    nova["n3_CC"]                = f"{cc['n3_cod']} {cc['n3_desc']}" if cc else ""
    nova["n4_cod_centro_custo"]  = cc["n4_cod"]       if cc else ""
    nova["n4_centro_custo"]      = cc["n4_desc"]      if cc else ""
    nova["n4_CC"]                = f"{cc['n4_cod']} {cc['n4_desc']}" if cc else ""

    linhas_saida.append(nova)

fechamento_df = pd.DataFrame(linhas_saida, columns=modelo_cols)

print(f"Linhas geradas: {len(fechamento_df)}")
print(f"Filiais sem mapeamento: {len(filiais_sem_mapa)}")
if filiais_sem_mapa:
    print(set(filiais_sem_mapa))
print()
fechamento_df.head(5)

## Salvando o arquivo de saida

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str((Path.cwd().parent / "Utitlities").resolve()))
from fechamento_excel import parse_valor_fechamento, parse_data_fechamento, gravar_fechamento_excel

import datetime

SHEET_NAME = 'ATUA_receitas'

arquivo_saida_exec = ARQUIVO_SAIDA
try:
    gravar_fechamento_excel(fechamento_df, arquivo_saida_exec, sheet_name=SHEET_NAME)
    print(f"Arquivo gerado: {arquivo_saida_exec.resolve()}")
    print(f"Linhas gravadas: {len(fechamento_df)}")
except PermissionError:
    ts = datetime.datetime.now().strftime("%H%M%S")
    alt = arquivo_saida_exec.with_stem(f"{arquivo_saida_exec.stem}_{ts}")
    gravar_fechamento_excel(fechamento_df, alt, sheet_name=SHEET_NAME)
    print(f"[AVISO] Arquivo principal em uso. Salvo como: {alt.resolve()}")
    print(f"Linhas gravadas: {len(fechamento_df)}")

if filiais_sem_mapa:
    print(f"[PENDENTE] {len(filiais_sem_mapa)} linha(s) com filial nao mapeada:")
    for f in sorted(set(filiais_sem_mapa)):
        n = filiais_sem_mapa.count(f)
        print(f"  '{f}' ({n} ocorrencia(s))")
else:
    print("[OK] Todas as filiais foram mapeadas. Nenhum item pendente.")